# TalentGraph: Production-Grade Agentic Resume Screening & Evaluation Engine

Modern enterprise recruitment requires screening candidates against diverse, complex job descriptions. Traditional keyword matching misses context, while **Naive Retrieval-Augmented Generation (RAG)** suffers from three severe flaws in document analysis:
1. **Information Fragmentation:** Document details (e.g., career summary vs. project-specific achievements) get split across unrelated chunks.
2. **Pool Coverage Blindspots:** Standard top-$k$ similarity retrieves multiple chunks from the same 2-3 resumes, silently dropping other qualified candidates across the applicant pool.
3. **Ungrounded Hallucinations:** Plain language models invent plausible details (like salary expectations or relocation willingness) when documents do not contain the answer.

This notebook builds a complete, end-to-end **Agentic RAG System** that progressively resolves each failure mode:
- **Small-to-Big Retrieval (Parent Document Pattern):** Indexes fine-grained chunks for high vector recall, but expands to complete candidate profiles for synthesis.
- **Guardrailed Structured Outputs (Pydantic):** Enforces evidence citations and explicit `unknowns` reporting.
- **Two-Stage Re-ranking:** Over-fetches candidate chunks and re-scores them with high-precision scoring to fix semantic drift.
- **Adaptive Query Routing:** Classifies recruiter intent (single-candidate deep-dive, multi-candidate comparison, broad pool screening, or unanswerable queries).
- **Corrective RAG (CRAG) & Self-Reflection:** Automatically grades retrieved context for relevance and rewrites queries when evidence is insufficient.
- **LangGraph Agentic Orchestration:** Wires the entire pipeline into an autonomous, self-correcting decision graph.

---


## Part 0 — Dependencies and Setup

We rely on modern, in-process tools for production prototyping:
- `langchain` & `langchain-openai`: Core LLM, embeddings, and chat wrappers.
- `langgraph`: Declarative state-machine orchestration for multi-agent workflows.
- `pydantic`: Schema validation and structured outputs.
- `pypdf`: PDF text extraction from candidate resumes.
- `python-dotenv`: Environment variable management.


In [ ]:
# Install required dependencies if not already present
# %pip install -qU langchain langchain-openai langgraph pydantic pypdf python-dotenv


### API Key & Configuration
We load our credentials safely from `.env` or prompt interactively if unset.


In [ ]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

# Model constants for swappability
MODEL_NAME = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"


In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
print(f"Initialized LLM ({MODEL_NAME}) and Embeddings ({EMBEDDING_MODEL}).")


## Part 1 — Ingestion & Document Processing

We maintain an applicant pool of 25 multi-domain tech resumes in `resumes/`:
- **Java Developers** (Full-stack, backend, enterprise J2EE)
- **DevOps Architects & Automation Leads**
- **QA & Mobile Test Automation Engineers**
- **Project Managers & Certified Scrum Masters (PMP, CSM)**
- **Technical Program Managers (TPM)**
- **Business Systems Analysts (BSA)**
- **Big Data / Hadoop Developers & PHP Specialists**

We extract raw text with `pypdf` and structure each applicant with explicit metadata.


In [ ]:
import os
from pypdf import PdfReader

RESUME_DIR = "resumes" if os.path.exists("resumes") else "../resumes"

CANDIDATES = [
    {"id": "mahesh_java", "file": "mahesh_java_developer.pdf", "name": "Mahesh", "role": "Sr. Full Stack Java Developer"},
    {"id": "sharath_java", "file": "sharath_java_developer.pdf", "name": "Sharath", "role": "Java Full Stack Developer"},
    {"id": "anudeep_java", "file": "anudeep_java_developer.pdf", "name": "Anudeep", "role": "Sr. Java Programmer"},
    {"id": "chetan_java", "file": "chetan_java_developer.pdf", "name": "Chetan Babu", "role": "Sr. Java Developer"},
    {"id": "pavan_java", "file": "pavan_java_developer.pdf", "name": "Pavan Kumar", "role": "Full Stack Java Developer"},
    {"id": "ashwini_java", "file": "ashwini_j2ee_developer.pdf", "name": "Ashwini", "role": "Sr. Java/J2EE Developer"},
    {"id": "sougandh_java", "file": "sougandh_java_developer.pdf", "name": "Sougandh", "role": "Java Fullstack Developer"},
    {"id": "nilesh_php", "file": "nilesh_php_lead.pdf", "name": "Nilesh Banthiya", "role": "PHP Lead / Web Developer"},
    {"id": "venkata_php", "file": "venkata_php_developer.pdf", "name": "Venkata", "role": "Sr. PHP / Drupal Developer"},
    {"id": "mani_hadoop", "file": "mani_hadoop_developer.pdf", "name": "Mani", "role": "Sr. Hadoop Developer"},
    {"id": "bapuji_hadoop", "file": "bapuji_hadoop_developer.pdf", "name": "Bapuji", "role": "Sr. Hadoop Developer"},
    {"id": "navneet_ba", "file": "navneet_business_analyst.pdf", "name": "Navneet", "role": "Business Analyst / Consultant"},
    {"id": "shail_ba", "file": "shail_business_analyst.pdf", "name": "Shail Tank", "role": "Business Analyst"},
    {"id": "krishna_bsa", "file": "krishna_business_systems_analyst.pdf", "name": "Krishna", "role": "Sr. Business Systems Analyst"},
    {"id": "amar_bsa", "file": "amar_business_systems_analyst.pdf", "name": "Amar", "role": "Business Systems Analyst"},
    {"id": "priyanka_bsa", "file": "priyanka_business_systems_analyst.pdf", "name": "Priyanka", "role": "Sr. Business Systems Analyst"},
    {"id": "avinash_pm", "file": "avinash_project_manager.pdf", "name": "Avinash", "role": "Project Manager"},
    {"id": "srivatsan_pm", "file": "srivatsan_project_manager.pdf", "name": "Srivatsan Ramabhadran", "role": "Project Manager / Business Data Analyst"},
    {"id": "adelina_pm", "file": "adelina_project_manager.pdf", "name": "Adelina Erimia", "role": "Project Manager (PMP)"},
    {"id": "adhi_scrum", "file": "adhi_scrum_master.pdf", "name": "Adhi Gopalam", "role": "Certified Scrum Master & Business Analyst"},
    {"id": "ranjan_pm_scrum", "file": "ranjan_pm_scrum_master.pdf", "name": "Ranjan Kumar Verma", "role": "Project Manager / Scrum Master"},
    {"id": "raju_tpm", "file": "raju_technical_program_manager.pdf", "name": "Raju Goduguchinta", "role": "Sr. Technical Program Manager"},
    {"id": "ravi_pm_devops", "file": "ravi_pm_devops.pdf", "name": "Ravi Prasad Burra", "role": "Certified PM / DevOps-Automation Architect"},
    {"id": "murali_pm_qa", "file": "murali_pm_qa.pdf", "name": "Muralidhar Chandrashekar", "role": "Project Manager (QA/Testing)"},
    {"id": "gautami_qa", "file": "gautami_qa_mobile_testing.pdf", "name": "Gautami Bulusu", "role": "QA / Mobile Test Engineer"},
]

def extract_pdf_text(path):
    if not os.path.exists(path):
        return ""
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

full_resumes = {}
for c in CANDIDATES:
    pdf_path = os.path.join(RESUME_DIR, c["file"])
    text = extract_pdf_text(pdf_path)
    full_resumes[c["id"]] = {"name": c["name"], "role": c["role"], "text": text}

print(f"Loaded {len(full_resumes)} candidate profiles.")
for cid, info in list(full_resumes.items())[:5]:
    print(f"  [{cid:16s}] {info['name']:22s} | {info['role']:38s} | {len(info['text']):5d} chars")


## Part 2 — Vector Indexing & Ingestion

We segment resumes into 800-character chunks with a 100-character overlap using `RecursiveCharacterTextSplitter`. Every chunk preserves rich metadata:
- `candidate_id`
- `name`
- `role`
- `chunk_index`

We maintain an in-memory vector store with disk caching to avoid redundant embedding API calls.


In [ ]:
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

CACHE_PATH = "cache/resume_vectorstore.json"
os.makedirs("cache", exist_ok=True)

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

def build_chunk_documents():
    docs = []
    for cid, info in full_resumes.items():
        chunks = splitter.split_text(info["text"])
        for idx, chunk in enumerate(chunks):
            docs.append(
                Document(
                    page_content=chunk,
                    metadata={
                        "candidate_id": cid,
                        "name": info["name"],
                        "role": info["role"],
                        "chunk_index": idx
                    }
                )
            )
    return docs

all_chunks = build_chunk_documents()
print(f"Generated {len(all_chunks)} chunks across {len(full_resumes)} candidate resumes.")

# Vector store initialization
vectorstore = InMemoryVectorStore(embeddings)

if os.path.exists(CACHE_PATH):
    try:
        with open(CACHE_PATH, "r") as f:
            cached_data = json.load(f)
        docs = [Document(page_content=d["text"], metadata=d["metadata"]) for d in cached_data["docs"]]
        vectorstore.add_documents(docs)
        print(f"Loaded {len(docs)} chunks from local cache ({CACHE_PATH}).")
    except Exception as e:
        print(f"Cache load skipped ({e}); embedding documents...")
        vectorstore.add_documents(all_chunks)
else:
    vectorstore.add_documents(all_chunks)
    try:
        to_cache = {"docs": [{"text": d.page_content, "metadata": d.metadata} for d in all_chunks]}
        with open(CACHE_PATH, "w") as f:
            json.dump(to_cache, f)
        print(f"Cached {len(all_chunks)} chunks to {CACHE_PATH}.")
    except Exception as e:
        print(f"Notice: Cache writing skipped: {e}")


## Part 3 — Baseline Naive RAG & Empirical Failure Analysis

A naive RAG pipeline simply embeds the recruiter's prompt, retrieves the top-$k$ nearest chunks, and prompts the LLM.


In [ ]:
NAIVE_SYSTEM_PROMPT = (
    "You are a recruitment assistant. Answer the recruiter's inquiry strictly using "
    "the resume excerpts provided in the context below."
)

def naive_rag(query, k=4):
    hits = vectorstore.similarity_search(query, k=k)
    print("Retrieved Chunks:")
    for h in hits:
        print(f"  - {h.metadata['name']} (chunk #{h.metadata['chunk_index']})")
    
    context = "\n\n".join(f"[{h.metadata['name']}]\n{h.page_content}" for h in hits)
    prompt = f"{NAIVE_SYSTEM_PROMPT}\n\nContext:\n{context}\n\nQuestion: {query}"
    answer = llm.invoke(prompt).content
    print("\nLLM Answer:\n" + answer)
    return hits, answer


### Test Case 1: Simple Role Matching
When a query targets a specific skill set, naive RAG performs reasonably well because the embedding of the query aligns directly with relevant chunks.


In [ ]:
_ = naive_rag(
    "We are hiring a senior backend engineer with strong Java/J2EE experience "
    "and exposure to cloud platforms like AWS. Who would you recommend and why?"
)


### Where Naive RAG Fails: Three Critical Vulnerabilities

#### Failure Mode 1: Information Fragmentation
A question requires two distinct facts from the same resume located in different sections (e.g., total years of experience from the executive summary + specific client project from the work history). Top-$k$ vector search frequently retrieves one chunk and misses the other.


In [ ]:
FRAGMENTATION_QUERY = "How many years of experience does Mahesh have in total, and which healthcare client did he work for?"
_ = naive_rag(FRAGMENTATION_QUERY)


#### Failure Mode 2: Pool Coverage Blindspots
When a recruiter asks to review or shortlist across the entire pool, top-$k$ search selects a handful of chunks from only 2-3 candidates, completely ignoring other qualified applicants.


In [ ]:
RUNNING_QUERY = (
    'Shortlist every candidate in the pool with project-management or Scrum-master '
    'experience for a "Technical Delivery Lead" panel review, and give evidence for each one you include.'
)

hits, answer = naive_rag(RUNNING_QUERY)
covered_ids = {h.metadata["candidate_id"] for h in hits}
missing_candidates = [c["name"] for c in CANDIDATES if c["id"] not in covered_ids]
print(f"\nCandidate Pool Coverage: {len(covered_ids)} / {len(full_resumes)} candidates represented.")
print(f"Candidates entirely omitted: {len(missing_candidates)}")


#### Failure Mode 3: Ungrounded Hallucination
When asked about facts absent from resumes (e.g., salary expectations, relocation preferences), naive generation tends to hallucinate or speculate rather than strictly admit missing data.


In [ ]:
_ = naive_rag("What is Mahesh's salary expectation, and would he be willing to relocate to another city?")


## Part 4 — Solving Fragmentation: Small-to-Big Retrieval (Parent Document Pattern)

**Core Principle:** *Search small, synthesize big.*
Vector embeddings work best on small, focused passages (800 chars). However, synthesis requires complete context. Instead of passing fragmented chunks to the LLM, we retrieve top chunks and trace them back to their complete parent resumes.


In [ ]:
def search_full_resumes(query, k=4):
    hits = vectorstore.similarity_search(query, k=k)
    # Deduplicate candidate IDs while preserving rank order
    seen_ids = list(dict.fromkeys(h.metadata["candidate_id"] for h in hits))
    print("Small-to-Big Retrieval -> Full Candidates:", [full_resumes[cid]["name"] for cid in seen_ids])
    
    return "\n\n".join(
        f"--- Candidate: {full_resumes[cid]['name']} ({full_resumes[cid]['role']}) ---\n{full_resumes[cid]['text']}"
        for cid in seen_ids
    )

def grounded_rag(query, k=4):
    context = search_full_resumes(query, k=k)
    prompt = f"{NAIVE_SYSTEM_PROMPT}\n\nContext:\n{context}\n\nQuestion: {query}"
    answer = llm.invoke(prompt).content
    print(f"Context payload: {len(context)} characters passed to LLM.")
    print("\nAnswer:\n" + answer)
    return context, answer

# Re-test the fragmentation query:
_ = grounded_rag(FRAGMENTATION_QUERY)


## Part 5 — Solving Hallucination: Guardrailed Structured Outputs

Small-to-big retrieval ensures complete context, but does not guarantee the LLM won't speculate on missing information.
We enforce a strict Pydantic schema with:
1. `match_score` (0-10)
2. `evidence` (mandatory verbatim quote or close paraphrase)
3. `unknowns` (explicit reporting of missing information, forbidding hallucination)


In [ ]:
from pydantic import BaseModel, Field

class CandidateAssessment(BaseModel):
    candidate_id: str = Field(description="Exact candidate identifier from context")
    name: str = Field(description="Candidate full name")
    match_score: int = Field(description="0 to 10 relevance fit score, 0=unrelated, 10=exceptional match")
    evidence: str = Field(description="Direct quote or factual excerpt from resume validating score")
    unknowns: str = Field(description="Explicitly state unmentioned attributes or 'none'")

class ScreeningReport(BaseModel):
    summary: str = Field(description="Executive summary of candidate assessment")
    candidates: list[CandidateAssessment]

structured_llm = llm.with_structured_output(ScreeningReport)

def screen_candidates(query, k=4):
    context = search_full_resumes(query, k=k)
    prompt = (
        f"{NAIVE_SYSTEM_PROMPT}\n"
        "Strict Rule: Rely solely on facts in the provided context. If information is missing, "
        "record it in the 'unknowns' field. Never assume or extrapolate.\n\n"
        f"Context:\n{context}\n\nRecruiter Query: {query}"
    )
    return structured_llm.invoke(prompt)

# Test ungrounded query under structured guardrails:
report = screen_candidates("What is Mahesh's salary expectation, and would he be willing to relocate to another city?")
print("Summary:", report.summary, "\n")
for c in report.candidates:
    print(f"Candidate: {c.name} | Score: {c.match_score}/10")
    print(f"  Evidence: {c.evidence}")
    print(f"  Unknowns: {c.unknowns}\n")


## Part 6 — Solving Precision: Two-Stage Retrieval (Re-ranking)

Vector similarity operates in embedding space, which measures topical proximity rather than exact factual match.
**Two-Stage Search Architecture:**
1. **Stage 1 (High Recall):** Over-fetch candidate chunks cheaply ($k=15$) using vector similarity.
2. **Stage 2 (High Precision):** Score each candidate chunk on a 0-10 relevance scale using an LLM cross-evaluator, retaining only the top-$k$ survivors.


In [ ]:
class ChunkScore(BaseModel):
    index: int = Field(description="The numeric index of the chunk being evaluated")
    score: int = Field(description="0-10 relevance score of chunk answering the query directly")

class RerankBatchResult(BaseModel):
    scores: list[ChunkScore]

reranker_llm = llm.with_structured_output(RerankBatchResult)

def rerank(query, chunks, top_n=4):
    chunk_listing = "\n\n".join(
        f"[{i}] ({c.metadata['name']} - {c.metadata['role']}): {c.page_content}"
        for i, c in enumerate(chunks)
    )
    prompt = (
        f"Recruiter Query: {query}\n\n"
        "Score each numbered resume chunk below from 0 to 10 based on whether it directly "
        "provides concrete evidence answering the query. Differentiate exact hands-on experience from passive mentions.\n\n"
        f"{chunk_listing}"
    )
    res = reranker_llm.invoke(prompt)
    score_map = {s.index: s.score for s in res.scores}
    ranked = sorted(enumerate(chunks), key=lambda x: score_map.get(x[0], 0), reverse=True)
    return [chunk for _, chunk in ranked[:top_n]]

RERANK_QUERY = "Which candidate has hands-on experience with automated testing frameworks like Selenium?"

print("--- Stage 1: Naive Vector Top-4 ---")
naive_hits = vectorstore.similarity_search(RERANK_QUERY, k=4)
print([f"{h.metadata['name']} (#{h.metadata['chunk_index']})" for h in naive_hits])

print("\n--- Stage 2: Over-fetch 15 & Re-rank to Top-4 ---")
overfetched = vectorstore.similarity_search(RERANK_QUERY, k=15)
reranked_hits = rerank(RERANK_QUERY, overfetched, top_n=4)
print([f"{h.metadata['name']} (#{h.metadata['chunk_index']})" for h in reranked_hits])


## Part 7 — Adaptive Query Routing (Adaptive RAG)

Different recruiter queries require completely different execution strategies:
1. `single_candidate_lookup`: Specific query about a known candidate (e.g. "Tell me about Mahesh's cloud background").
2. `candidate_comparison`: Side-by-side comparison between multiple named applicants.
3. `pool_screening`: Broad evaluation matching a job description across the entire pool.
4. `out_of_scope`: Inquiries unrelated to candidate resumes or hiring.

An **Adaptive Router** inspects the query and directs it to the optimal retrieval pipeline.


In [ ]:
from typing import Literal

class QueryRoute(BaseModel):
    route: Literal["single_candidate_lookup", "candidate_comparison", "pool_screening", "out_of_scope"]
    candidate_names: list[str] = Field(default_factory=list, description="Target candidate names mentioned, if any")
    cleaned_query: str = Field(description="Search-optimized reformulation of the recruiter query")

router_llm = llm.with_structured_output(QueryRoute)

def route_recruiter_query(query: str) -> QueryRoute:
    prompt = (
        "Classify the following recruiter query into one of four routing strategies:\n"
        "1. single_candidate_lookup: targeting a specific named candidate.\n"
        "2. candidate_comparison: comparing two or more specific named candidates.\n"
        "3. pool_screening: searching the candidate pool for specific skills, roles, or qualifications.\n"
        "4. out_of_scope: unrelated to hiring, resumes, or candidates.\n\n"
        f"Query: {query}"
    )
    return router_llm.invoke(prompt)

# Test the router across distinct intents:
for sample_q in [
    "What are Sharath's cloud certifications and backend stack?",
    "Compare Mahesh and Sougandh on their testing automation background.",
    "Find candidates with strong Agile Scrum Master experience.",
    "What is the weather in San Francisco today?"
]:
    decision = route_recruiter_query(sample_q)
    print(f"Query: '{sample_q}'\n  -> Route: {decision.route} | Targets: {decision.candidate_names}\n")


## Part 8 — Corrective RAG (CRAG) & Self-Reflection

Even with re-ranking, vector search can retrieve irrelevant chunks if the query terms are ambiguous.
**Corrective RAG (CRAG)** introduces a self-reflection loop:
1. **Retrieval Grader:** Evaluates whether retrieved evidence is sufficient and relevant to answer the inquiry.
2. **Query Transformation / Rewriter:** If retrieved documents are graded as *insufficient*, the system rewrites the search query to emphasize key technical terms and retries retrieval.


In [ ]:
class RetrievalGrade(BaseModel):
    is_sufficient: bool = Field(description="True if context has sufficient factual detail to answer the query")
    relevance_score: int = Field(description="0-10 confidence score of retrieved evidence relevance")
    explanation: str = Field(description="Brief assessment of missing or present facts")

grader_llm = llm.with_structured_output(RetrievalGrade)

def grade_retrieval(query: str, context: str) -> RetrievalGrade:
    prompt = (
        f"Recruiter Query: {query}\n\n"
        f"Retrieved Context:\n{context}\n\n"
        "Determine if the retrieved context contains sufficient, relevant facts to answer the recruiter's question. "
        "Grade strictly based on factual presence, not general plausibility."
    )
    return grader_llm.invoke(prompt)

def rewrite_query(original_query: str, critique: str) -> str:
    prompt = (
        f"Original Recruiter Query: {original_query}\n"
        f"Critique of Initial Retrieval: {critique}\n\n"
        "Formulate a more specific, keyword-dense search query targeting the missing technical skills and candidate qualifications."
    )
    return llm.invoke(prompt).content.strip()

# Demonstrate grading:
sample_context = full_resumes["mahesh_java"]["text"][:1000]
grade = grade_retrieval("Does Mahesh have hands-on experience deploying Kubernetes in production?", sample_context)
print(f"Grading Result: Sufficient={grade.is_sufficient} | Score={grade.relevance_score}/10")
print(f"Explanation: {grade.explanation}")


## Part 9 — Full Multi-Agent Orchestration with LangGraph

Now we integrate all components into an autonomous, self-correcting state machine using **LangGraph**:

```
[Start] --> [Route Query]
                 |
        +--------+--------+
        |                 |
  (Out of Scope)    (Candidate Queries)
        |                 |
   [Direct Msg]    [Retrieve & Re-rank]
                          |
                    [Grade Context]
                          |
             +------------+------------+
             |                         |
      (Grade < 6 / Retry)       (Grade >= 6)
             |                         |
     [Rewrite Query]            [Generate Report]
             |                         |
             +---> [Retrieve]          +---> [End]
```


In [ ]:
from typing import TypedDict, Annotated, Optional
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    query: str
    current_search_query: str
    route: str
    target_candidates: list[str]
    context: str
    retrieval_grade: Optional[RetrievalGrade]
    rewrite_attempts: int
    report: Optional[ScreeningReport]
    direct_response: Optional[str]

def router_node(state: AgentState):
    print(f"\n--- [NODE] Routing Query: '{state['query']}' ---")
    decision = route_recruiter_query(state["query"])
    return {
        "route": decision.route,
        "target_candidates": decision.candidate_names,
        "current_search_query": decision.cleaned_query,
        "rewrite_attempts": 0
    }

def retrieve_node(state: AgentState):
    print(f"--- [NODE] Retrieving & Re-ranking for: '{state['current_search_query']}' ---")
    route = state["route"]
    
    # Targeted candidate lookup optimization
    if route == "single_candidate_lookup" and state.get("target_candidates"):
        matched_cids = [
            cid for cid, info in full_resumes.items()
            if any(name.lower() in info["name"].lower() for name in state["target_candidates"])
        ]
        if matched_cids:
            context = "\n\n".join(
                f"--- {full_resumes[cid]['name']} ({full_resumes[cid]['role']}) ---\n{full_resumes[cid]['text']}"
                for cid in matched_cids
            )
            return {"context": context}
            
    # Two-stage Small-to-Big Retrieval with Re-ranking
    overfetched = vectorstore.similarity_search(state["current_search_query"], k=15)
    reranked = rerank(state["current_search_query"], overfetched, top_n=4)
    seen_ids = list(dict.fromkeys(c.metadata["candidate_id"] for c in reranked))
    
    context = "\n\n".join(
        f"--- {full_resumes[cid]['name']} ({full_resumes[cid]['role']}) ---\n{full_resumes[cid]['text']}"
        for cid in seen_ids
    )
    return {"context": context}

def grade_node(state: AgentState):
    print("--- [NODE] Evaluating Retrieval Quality (CRAG) ---")
    grade = grade_retrieval(state["query"], state["context"])
    print(f"  Score: {grade.relevance_score}/10 | Sufficient: {grade.is_sufficient}")
    return {"retrieval_grade": grade}

def rewrite_node(state: AgentState):
    attempts = state.get("rewrite_attempts", 0) + 1
    critique = state["retrieval_grade"].explanation if state.get("retrieval_grade") else "Low relevance score"
    print(f"--- [NODE] Self-Correction: Rewriting Query (Attempt {attempts}) ---")
    new_query = rewrite_query(state["query"], critique)
    print(f"  New Reformulated Query: '{new_query}'")
    return {
        "current_search_query": new_query,
        "rewrite_attempts": attempts
    }

def generate_node(state: AgentState):
    print("--- [NODE] Synthesizing Guardrailed Screening Report ---")
    prompt = (
        f"{NAIVE_SYSTEM_PROMPT}\n"
        "Strict Rule: Rely solely on facts present in the context. Record any unstated attributes in 'unknowns'.\n\n"
        f"Context:\n{state['context']}\n\nRecruiter Query: {state['query']}"
    )
    report = structured_llm.invoke(prompt)
    return {"report": report}

def out_of_scope_node(state: AgentState):
    print("--- [NODE] Handling Out-of-Scope Query ---")
    response = (
        "This system is specialized for candidate resume screening and technical evaluation. "
        "The requested query is outside the scope of the recruitment applicant pool."
    )
    return {"direct_response": response}

# Conditional Routing Logic
def route_after_classifier(state: AgentState):
    if state["route"] == "out_of_scope":
        return "out_of_scope_node"
    return "retrieve_node"

def route_after_grader(state: AgentState):
    grade = state.get("retrieval_grade")
    attempts = state.get("rewrite_attempts", 0)
    # If grade is insufficient and we haven't exceeded retry limit, rewrite and search again
    if grade and not grade.is_sufficient and attempts < 1:
        return "rewrite_node"
    return "generate_node"

# Build LangGraph
workflow = StateGraph(AgentState)

workflow.add_node("router_node", router_node)
workflow.add_node("retrieve_node", retrieve_node)
workflow.add_node("grade_node", grade_node)
workflow.add_node("rewrite_node", rewrite_node)
workflow.add_node("generate_node", generate_node)
workflow.add_node("out_of_scope_node", out_of_scope_node)

workflow.set_entry_point("router_node")
workflow.add_conditional_edges("router_node", route_after_classifier)
workflow.add_edge("retrieve_node", "grade_node")
workflow.add_conditional_edges("grade_node", route_after_grader)
workflow.add_edge("rewrite_node", "retrieve_node")
workflow.add_edge("generate_node", END)
workflow.add_edge("out_of_scope_node", END)

agent_app = workflow.compile()
print("Successfully compiled LangGraph Agentic RAG Workflow.")


### Running End-to-End Agent Execution

Let's execute the full self-correcting agent on our benchmark challenges.


In [ ]:
def run_agentic_screening(query_text: str):
    initial_state = {
        "query": query_text,
        "current_search_query": query_text,
        "route": "",
        "target_candidates": [],
        "context": "",
        "retrieval_grade": None,
        "rewrite_attempts": 0,
        "report": None,
        "direct_response": None
    }
    
    final_output = agent_app.invoke(initial_state)
    
    if final_output.get("direct_response"):
        print("\n=== Response ===")
        print(final_output["direct_response"])
        return
        
    report = final_output.get("report")
    if report:
        print("\n================ FINAL SCREENING REPORT ================")
        print(f"Executive Summary: {report.summary}\n")
        for c in report.candidates:
            print(f"• Candidate: {c.name} (Fit Score: {c.match_score}/10)")
            print(f"  Key Evidence: {c.evidence}")
            print(f"  Identified Gaps/Unknowns: {c.unknowns}\n")

# Run complex real-world query:
run_agentic_screening(
    "Shortlist candidates with automated mobile testing and Selenium experience, "
    "and identify any missing information regarding their cloud exposure."
)


## Part 10 — System Performance & Architectural Benchmark

| Feature / Metric | Naive RAG | Small-to-Big RAG | Reranked RAG | **Agentic RAG (LangGraph)** |
| :--- | :---: | :---: | :---: | :---: |
| **Context Fragmentation** | Fails (Chunk splits) | **Resolved** (Parent doc) | **Resolved** | **Resolved** |
| **Hallucination Protection** | None (Speculates) | Weak | Moderate | **Strong** (Guardrailed Pydantic Schema) |
| **Precision on Technical Skills** | Low (Cosine proxy) | Moderate | **High** (Two-stage re-rank) | **Very High** |
| **Query Flexibility** | Fixed top-$k$ only | Fixed top-$k$ only | Fixed top-$k$ only | **Adaptive Routing + Self-Correction** |
| **Self-Reflection & Recovery** | None | None | None | **Active (CRAG Grader + Query Rewriter)** |
| **Out-of-Scope Handling** | Hallucinates context | Hallucinates context | Hallucinates context | **Guarded via Router Classification** |

### Conclusion
By transitioning from **Naive RAG** to an **Agentic System**, we transform brittle similarity searches into a reliable, enterprise-grade talent assessment engine that produces auditable, citation-backed candidate reports.
